# 01 — Data Cleaning & Preparation

Clean, auditable transaction frames for the rest of the analysis.

**What this notebook does**
- Load the UCI Online Retail file
- Cast types and extract temporal features
- Isolate cancellations, anomalies, and guest checkouts
- Create `LineRevenue` and `Is_UK`
- Persist every intermediate frame under `data/`


In [9]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.cleaning import full_pipeline, load_raw, structural_cleansing

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")

print("Working directory:", ROOT)

Working directory: C:\Users\ahmed\Downloads\online_retail_capstone


## 1. Load & first look

Each row is a *line* inside an invoice, not a complete order. That distinction matters later when we calculate basket size and frequency.

In [10]:
raw = load_raw(ROOT / "data" / "Online_Retail.xlsx")
print(f"Raw shape: {raw.shape}")
print(raw.dtypes)
raw.head(3)

Raw shape: (541909, 8)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom


In [11]:
print("Missing values:")
print(raw.isna().sum())
print(f"\nCustomerID missing rate: {raw['CustomerID'].isna().mean():.1%}")
print(f"Date range: {raw['InvoiceDate'].min()} → {raw['InvoiceDate'].max()}")
print("\nTop countries:")
print(raw["Country"].value_counts().head(8))

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

CustomerID missing rate: 24.9%
Date range: 2010-12-01 08:26:00 → 2011-12-09 12:50:00

Top countries:
Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Name: count, dtype: int64


## 2. Structural cleansing

- `InvoiceDate` → proper datetime + Year / Month / DayOfWeek / Hour
- String columns stripped
- Quantity & UnitPrice forced to numeric

Rows that lack `CustomerID` are kept. Guest checkouts still generate revenue; isolating them later is cleaner than deleting them now.

In [12]:
structured = structural_cleansing(raw)
structured[["InvoiceDate", "Year", "Month", "DayOfWeek", "Hour"]].head()

,InvoiceDate,Year,Month,DayOfWeek,Hour
0,2010-12-01 08:26:00,2010,12,Wednesday,8
1,2010-12-01 08:26:00,2010,12,Wednesday,8
2,2010-12-01 08:26:00,2010,12,Wednesday,8
3,2010-12-01 08:26:00,2010,12,Wednesday,8
4,2010-12-01 08:26:00,2010,12,Wednesday,8


## 3. Transactional validation

Three separate buckets:

1. **Cancellations** – InvoiceNo starts with “C” (negative quantities)
2. **Anomalies** – zero or negative UnitPrice / Quantity on non-cancel rows
3. **Guests** – genuine sales that simply lack a CustomerID

Everything that remains becomes the *registered* frame used for RFM.

In [13]:
result = full_pipeline(ROOT / "data" / "Online_Retail.xlsx")

for name in ["registered", "guests", "cancellations", "anomalies"]:
    df = result[name]
    print(f"{name:15s}: {df.shape[0]:>8,} rows | revenue £{df['LineRevenue'].sum():>12,.0f}")

registered     :  397,884 rows | revenue £   8,911,408
guests         :  132,220 rows | revenue £   1,755,277
cancellations  :    9,288 rows | revenue £    -896,812
anomalies      :    2,517 rows | revenue £     -22,124


In [14]:
reg = result["registered"]
print("Registered – Quantity stats")
print(reg["Quantity"].describe())
print("\nAny remaining negative prices?", (reg["UnitPrice"] <= 0).any())
print("Any remaining negative qty?", (reg["Quantity"] <= 0).any())
print("CustomerID nulls left?", reg["CustomerID"].isna().sum())

Registered – Quantity stats
count   397884.00
mean        12.99
std        179.33
min          1.00
25%          2.00
50%          6.00
75%         12.00
max      80995.00
Name: Quantity, dtype: float64

Any remaining negative prices? False
Any remaining negative qty? False
CustomerID nulls left? 0


## 4. Feature generation

- `LineRevenue = Quantity × UnitPrice`
- `Is_UK` binary flag (dataset is ~91 % UK)

In [15]:
reg[["Quantity", "UnitPrice", "LineRevenue", "Is_UK"]].sample(5, random_state=1)

,Quantity,UnitPrice,LineRevenue,Is_UK
469745,12,1.25,15.00,0
233490,48,1.25,60.00,0
401578,2,2.46,4.92,1
172343,1,21.95,21.95,1
276445,12,0.65,7.80,1


## 5. Persist the frames

In [16]:
out = ROOT / "data"
result["registered"].to_csv(out / "registered_clean.csv", index=False)
result["guests"].to_csv(out / "guests.csv", index=False)
result["cancellations"].to_csv(out / "cancellations.csv", index=False)
result["anomalies"].to_csv(out / "anomalies.csv", index=False)
print("All frames written to data/")

All frames written to data/


### Decision log

| Decision | Reason |
|---|---|
| Keep guest rows separate | Revenue from anonymous buyers stays visible; RFM stays clean |
| Isolate “C” invoices | Return rate & product-level return analysis become possible |
| Isolate zero/negative price | Prevents distortion of average order value and monetary scores |
| Add temporal columns early | Seasonal, weekday and hourly patterns available for every later chart |